# ⚛️ 30 · From maps to moving atoms: molecular model series with Gaussian mixtures and neural networks
### One Gaussian per atom, chemistry as a restraint, and a whole family of atomic models along a motion

**BiologyToResearch** · Part F — Conformational heterogeneity: Gaussian mixtures and neural fields · Module 30 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/30_atomic_models_model_series.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 3–4 min to run everything); a GPU is optional |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-learn, PyTorch |
| **Prerequisites** | [25 · Gaussian mixtures](25_gaussian_mixture_molecules.ipynb) (GMM projection, FRC); [26–27](26_e2gmm_heterogeneity.ipynb) (latent spaces); [02 · Geometry](02_rotations_geometry.ipynb) (vectors, angles); [01 · From atoms to density](01_python_arrays_density_maps.ipynb) |
| **Estimated time** | 110 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Chen 2025 (molecular model series with GMMs and DNNs, *Commun Biol*); Chen, Toader & Lederman 2023 (models inside deep GMMs, *J Mol Biol*) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small, controlled *toy versions* of published methods (a 36-residue backbone model, simulated 4 Å maps,
> minutes of CPU). They illustrate the core ideas; they do **not** reproduce the paper's numbers or its full
> stereochemistry (side chains, rotamers, RNA). **All molecules and maps are generated by code** (nothing is downloaded);
> the Ramachandran reference used here is a synthetic stand-in, not statistics from the PDB.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Build** a protein backbone from torsion angles (φ, ψ) and **describe** a model as one Gaussian per atom.
2. **Compute** the stereochemistry a model must respect — bond lengths, bond angles, clashes, the Ramachandran plot —
   and **score** φ/ψ pairs with a 2D Gaussian mixture.
3. **Refine** atomic coordinates into a map by maximising the FRC of projections, and **show** why chemistry restraints
   are essential and how their weight trades fit against geometry.
4. **Explain** the staged ("boosting-like") refinement of Chen (2025): patches, then residues, then atoms.
5. **Train** a network that turns a latent coordinate into a full atomic model, fitted to a *series* of maps, and
   **verify** that it produces chemically sound models between the maps — unlike naïve interpolation.
6. **Name** what such model series can and cannot tell a biologist.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | tools, PyTorch | 5 min |
| 1 · A backbone from torsion angles | helix–turn–helix with a hinge; five conformations and their maps | 15 min |
| 2 · What chemistry allows | bonds, angles, clashes, Ramachandran GMM | 15 min |
| 3 · Refining atoms into a map | FRC of projections + geometry; the weight trade-off | 20 min |
| 4 · Patches, residues, atoms | staged refinement | 10 min |
| 5 · A model series | latent → atoms, trained on five maps; check between the maps | 20 min |
| 6 · What a model series tells you | interpretation and limits | 5 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A heterogeneity analysis (Modules 26–28) gives a *movie of densities*. But biologists reason with **atoms**: which
residue touches which, how far a gating helix moves, whether a salt bridge breaks. Building one atomic model into one
map is already skilled work; building a model for every frame of a movie, each chemically correct and consistent with
its neighbours, is the problem Muyuan Chen tackled in 2025: represent each atom by a Gaussian, let small neural networks
move the atoms, fit the maps' projections, and keep the chemistry right with restraints — for single maps and for whole
**model series** along a conformational change (for example the TRPV1 ion channel and a spliceosome).

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from scipy import ndimage
from sklearn.mixture import GaussianMixture

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

Helpers copied from [`course_toolkit.py`](../course_toolkit.py): `build_backbone` (the NeRF chain builder behind the
course's code-built molecules — Part F's flexible protein and the offline stand-ins; Module 02), `gmm_to_map`,
rotations, `lowpass`.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def gmm_to_map(centres, amps, sigmas, box: int, voxel: float = 1.0) -> np.ndarray:
    """Density [z, y, x] (box³, origin at the box centre) of Σ_j A_j exp(−|x − c_j|² / 2σ_j²); c and σ in Å."""
    centres = np.asarray(centres, float)
    s = np.broadcast_to(np.asarray(sigmas, float), (len(centres),))
    g = (np.arange(box) - (box - 1) / 2.0) * voxel
    ex = [np.exp(-(g[None, :] - centres[:, a, None]) ** 2 / (2 * s[:, None] ** 2)) for a in range(3)]
    return np.einsum("n,nz,ny,nx->zyx", np.asarray(amps, float), ex[2], ex[1], ex[0], optimize=True).astype(np.float32)
print("🧰 toolkit loaded")

---
## 1 · A backbone from torsion angles

> 📍 **Where we are.** Modules 26–29 worked with tens to hundreds of Gaussians standing for chunks of protein. Now each
> Gaussian is **one atom** (Chen 2025), and the result is an atomic model. This closes Part F.

A protein backbone is a chain N–Cα–C–N–Cα–C… with nearly fixed bond lengths and angles. Its shape is set almost entirely
by two **torsion angles** per residue: φ (rotation about N–Cα) and ψ (about Cα–C) — the ω torsion of the peptide bond is
held flat at 180°. The toolkit's `build_backbone` places atom after atom from these angles (the **NeRF** method): all
α-helices of the course were built with φ = −57.8°, ψ = −47.0°.

Our test molecule is a 36-residue **helix–turn–helix**: two 16-residue helices joined by a 4-residue turn. Changing one
ψ angle of the turn by Δ swings the second helix — a hinge motion. We build five conformations (Δ = −40°, −20°, 0°,
+20°, +40°), keeping the atoms N, Cα, C, O and Cβ of each residue (no further side chains in this toy).

In [ ]:
# ── 1.1 Build the helix–turn–helix at five hinge angles ──
H1, H2 = 16, 16
TURN_PHI, TURN_PSI = [-65.0, -120.0, 60.0, -75.0], [140.0, 120.0, 40.0, 150.0]
ATOM_NAMES = ["N", "CA", "C", "O", "CB"]
ATOM_Z = np.array([7, 6, 6, 8, 6], np.float32)                            # electrons per atom (Module 01)


def torsions(delta: float):
    """φ, ψ (degrees) of all residues; the last turn residue's ψ is changed by delta (the hinge)."""
    phi = np.r_[np.full(H1, -57.8), TURN_PHI, np.full(H2, -57.8)]
    psi = np.r_[np.full(H1, -47.0), TURN_PSI[:-1], [TURN_PSI[-1] + delta], np.full(H2, -47.0)]
    return phi, psi


def model(delta: float) -> np.ndarray:
    """Backbone (+Cβ) coordinates (n_res, 5, 3) in Å, centred on the Cα atoms."""
    bb = build_backbone(*torsions(delta))
    return bb - bb[:, 1].mean(0)


DELTAS = [-40.0, -20.0, 0.0, 20.0, 40.0]
models = {d: model(d) for d in DELTAS}
N_RES = len(models[0.0])
print(f"{N_RES} residues × 5 atoms = {N_RES * 5} atoms; Cα RMSD between Δ = −40° and +40°: "
      f"{np.sqrt(((models[-40.0][:, 1] - models[40.0][:, 1]) ** 2).sum(1).mean()):.1f} Å")
fig, ax = plt.subplots(figsize=(4.8, 4))
for d, col in zip(DELTAS, plt.cm.coolwarm(np.linspace(0, 1, 5))):
    ax.plot(models[d][:, 1, 0], models[d][:, 1, 1], "-o", ms=2, color=col, label=f"Δ = {d:+.0f}°")
ax.set_aspect("equal"); ax.set_xlabel("x (Å)"); ax.set_ylabel("y (Å)"); ax.legend(fontsize=7)
ax.set_title("Cα traces of five hinge conformations", fontsize=9); plt.tight_layout(); plt.show()

### 1.2 Maps, and comparing a model with a map by projections

For each conformation we make a **4 Å map** (Gaussians of 1.6 Å per atom on a 48³ grid of 1.5 Å voxels, low-passed to
4 Å) — standing in for the series of maps a heterogeneity analysis would produce. Chen (2025) compares a model with a map
**through projections**: project the map in many directions once, project the model's Gaussians the same way, and score
with the FRC (Module 25) up to the map's resolution. We precompute 200 projections of every map.

In [ ]:
# ── 1.2 Five 4 Å maps and 200 projections of each ──
N, PX, SIG_ATOM, RES_A = 48, 1.5, 1.6, 4.0
Z_ALL = np.tile(ATOM_Z, N_RES)
VIEWS = np.array([random_rotation(rng) for _ in range(200)]).astype(np.float32)
t0 = time.time()
maps, map_projs = {}, {}
for d in DELTAS:
    maps[d] = lowpass(gmm_to_map(models[d].reshape(-1, 3), Z_ALL, SIG_ATOM, N, PX), PX / RES_A)
    map_projs[d] = torch.tensor(np.array([rotate_volume(maps[d], R).sum(0) for R in VIEWS]), dtype=torch.float32)
print(f"maps and projections in {time.time() - t0:.0f} s")
fig, axes = plt.subplots(1, 5, figsize=(11, 2.4))
for ax, d in zip(axes, DELTAS):
    ax.imshow(maps[d].sum(0)); ax.set_title(f"map, Δ = {d:+.0f}°", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── 1.3 PyTorch: project one-Gaussian-per-atom models and score them with the FRC ──
GRID_T = torch.tensor((np.arange(N) - (N - 1) / 2) * PX, dtype=torch.float32)
Z_T = torch.tensor(Z_ALL)
VIEWS_T = torch.tensor(VIEWS)
_fy, _fx = np.fft.fftfreq(N), np.fft.rfftfreq(N)
RING_T = torch.tensor(np.minimum((np.sqrt(_fy[:, None] ** 2 + _fx[None, :] ** 2) * N).astype(int), N // 2).ravel())
RMAX = int(N * PX / RES_A)                                                 # FRC up to the map's resolution


def project_atoms(xyz, R):
    """Projections (b, N, N) of a model (atoms × 3, Å; one Gaussian of width SIG_ATOM per atom) at poses R."""
    p = torch.einsum("bij,nj->bni", R, xyz)
    gx = torch.exp(-(GRID_T - p[..., 0:1]) ** 2 / (2 * SIG_ATOM ** 2))
    gy = torch.exp(-(GRID_T - p[..., 1:2]) ** 2 / (2 * SIG_ATOM ** 2))
    return torch.einsum("bny,bnx->byx", gy * Z_T[None, :, None], gx)


def frc_torch(a, b, rmin: int, rmax: int):
    """Per-image mean Fourier ring correlation over rings rmin..rmax (Module 25)."""
    Fa = torch.fft.rfft2(a).reshape(a.shape[0], -1)
    Fb = torch.fft.rfft2(b).reshape(b.shape[0], -1)
    ring_sum = lambda v: torch.zeros(a.shape[0], N // 2 + 1).index_add(1, RING_T, v)
    num = ring_sum((Fa * Fb.conj()).real)
    return (num / torch.sqrt(ring_sum(Fa.abs() ** 2) * ring_sum(Fb.abs() ** 2) + 1e-12))[:, rmin:rmax + 1].mean(1)


def map_fit(xyz, delta: float, n_views: int = 32):
    """FRC between the model's projections and those of the map of conformation delta, on random views."""
    idx = torch.randint(0, len(VIEWS), (n_views,))
    return frc_torch(project_atoms(xyz, VIEWS_T[idx]), map_projs[delta][idx], 1, RMAX).mean()


x_start = torch.tensor(models[0.0].reshape(-1, 3), dtype=torch.float32)
x_target = torch.tensor(models[40.0].reshape(-1, 3), dtype=torch.float32)
torch.manual_seed(SEED)
print(f"fit of the Δ = 0° model to its own map: FRC {map_fit(x_start, 0.0).item():.3f}; "
      f"to the Δ = +40° map: {map_fit(x_start, 40.0).item():.3f}")

🔍 The model matches its own map almost perfectly and the other conformation's map clearly less well — the FRC of
projections "sees" the hinge. Next: what must a model respect besides the map?

---
## 2 · What chemistry allows

Chen (2025) scores a model's chemistry with statistical restraints:

- **bond lengths and angles** — a Gaussian likelihood around ideal values (Engh & Huber), with an extra penalty for
  outliers beyond 4.5 standard deviations;
- **planarity** of peptide bonds and other flat groups (dihedral thresholds of 30° and 10°);
- **clashes** — non-bonded atoms closer than the sum of their van der Waals radii minus 0.4 Å (with 0.4 Å more allowed for
  hydrogen-bond pairs);
- the **Ramachandran plot** — φ/ψ pairs scored by 2D Gaussian mixtures fitted to high-quality structures (six residue
  classes, 2,500 Gaussians each), as $\ln(R+C)$ with $R$ the mixture density and $C=e^{-10}$ a floor;
- **rotamers** of side chains (mixtures with 4, 25, 64 or 160 Gaussians for 1–4 χ angles) and an RNA suite library.

Our toy keeps bonds, angles, a simple clash term and a Ramachandran mixture. Let us compute them.

In [ ]:
# ── 2.1 Bonds, angles, clashes and backbone torsions of a model ──
IDX = lambda r, a: 5 * r + a                                              # atom index: residue r, atom a (0 N … 4 CB)
BOND_IDEAL = {"N-CA": 1.458, "CA-C": 1.525, "C-O": 1.231, "CA-CB": 1.530, "C-N": 1.329}
ANGLE_IDEAL = {"N-CA-C": 111.2, "CA-C-O": 120.5, "CA-C-N": 116.2, "C-N-CA": 121.7}
bonds, b0, angles, a0 = [], [], [], []
for r in range(N_RES):
    for (i, j), key in [((0, 1), "N-CA"), ((1, 2), "CA-C"), ((2, 3), "C-O"), ((1, 4), "CA-CB")]:
        bonds.append((IDX(r, i), IDX(r, j))); b0.append(BOND_IDEAL[key])
    angles += [(IDX(r, 0), IDX(r, 1), IDX(r, 2)), (IDX(r, 1), IDX(r, 2), IDX(r, 3))]; a0 += [111.2, 120.5]
    if r + 1 < N_RES:
        bonds.append((IDX(r, 2), IDX(r + 1, 0))); b0.append(BOND_IDEAL["C-N"])
        angles += [(IDX(r, 1), IDX(r, 2), IDX(r + 1, 0)), (IDX(r, 2), IDX(r + 1, 0), IDX(r + 1, 1))]; a0 += [116.2, 121.7]
BONDS, B0 = torch.tensor(bonds), torch.tensor(b0)
ANGLES, A0 = torch.tensor(angles), torch.tensor(np.radians(a0), dtype=torch.float32)
res_of, atom_of = np.repeat(np.arange(N_RES), 5), np.tile(np.arange(5), N_RES)
FAR = torch.tensor(np.abs(res_of[:, None] - res_of[None]) >= 3)          # pairs ≥ 3 residues apart
NO_PAIR = torch.tensor(((atom_of[:, None] == 0) & (atom_of[None] == 3)) | ((atom_of[:, None] == 3) & (atom_of[None] == 0)))
CLASH_D = torch.where(NO_PAIR, torch.tensor(2.4), torch.tensor(2.8))     # crude thresholds (Å); N–O may H-bond


def dihedral(p0, p1, p2, p3):
    """Torsion angle (radians) of the four points (…, 3)."""
    b0_, b1_, b2_ = p0 - p1, p2 - p1, p3 - p2
    b1n = b1_ / b1_.norm(dim=-1, keepdim=True)
    v = b0_ - (b0_ * b1n).sum(-1, keepdim=True) * b1n
    w = b2_ - (b2_ * b1n).sum(-1, keepdim=True) * b1n
    return torch.atan2((torch.cross(b1n, v, dim=-1) * w).sum(-1), (v * w).sum(-1))


def phi_psi(xyz):
    """φ and ψ (radians) of residues 1 … n−2 from coordinates (5n, 3)."""
    a = xyz.reshape(N_RES, 5, 3)
    phi = dihedral(a[:-2, 2], a[1:-1, 0], a[1:-1, 1], a[1:-1, 2])             # C(i−1), N, CA, C
    psi = dihedral(a[1:-1, 0], a[1:-1, 1], a[1:-1, 2], a[2:, 0])              # N, CA, C, N(i+1)
    return phi, psi


def geometry_report(xyz) -> dict:
    """Bond and angle RMS deviations from ideal values and the number of clashes."""
    with torch.no_grad():
        bl = (xyz[BONDS[:, 0]] - xyz[BONDS[:, 1]]).norm(dim=1)
        v1, v2 = xyz[ANGLES[:, 0]] - xyz[ANGLES[:, 1]], xyz[ANGLES[:, 2]] - xyz[ANGLES[:, 1]]
        ang = torch.acos(((v1 * v2).sum(1) / (v1.norm(dim=1) * v2.norm(dim=1))).clamp(-1, 1))
        D = (xyz[:, None] - xyz[None]).norm(dim=2)
        clashes = int(((D < CLASH_D) & FAR).sum().item() / 2)
    return dict(bond_rmsd=(bl - B0).pow(2).mean().sqrt().item(),
                angle_rmsd_deg=math.degrees((ang - A0).pow(2).mean().sqrt().item()), clashes=clashes)


print("ideal Δ = 0° model:", geometry_report(x_start))
ph, ps = phi_psi(x_start)
print("φ/ψ of the turn residues (degrees):", np.round(np.degrees(torch.stack([ph, ps], 1).numpy()[H1 - 1:H1 + 4]), 0).tolist())

### 2.2 A Ramachandran plot as a Gaussian mixture

The paper's Ramachandran mixtures were fitted to the φ/ψ angles of high-quality structures. To keep this notebook
self-contained we fit a mixture (scikit-learn, Module 04) to a **synthetic** reference sample placed on the textbook
allowed regions — right-handed α-helix around (−63°, −43°), β-sheet around (−120°, +130°), polyproline II around
(−75°, +145°) and the left-handed helix around (+57°, +47°). It is a stand-in that shows the mechanics, not real statistics.

In [ ]:
# ── 2.2 A synthetic Ramachandran reference and its 2D Gaussian mixture ──
regions = [((-63, -43), (10, 10), 0.45), ((-120, 130), (18, 15), 0.25), ((-75, 145), (10, 12), 0.15),
           ((57, 47), (10, 10), 0.05), ((-90, 0), (20, 20), 0.10)]
ref = np.concatenate([rng.normal(m, s, size=(int(4000 * w), 2)) for m, s, w in regions])
rama_gmm = GaussianMixture(6, covariance_type="full", random_state=SEED).fit(np.radians(ref))
RAMA_MU = torch.tensor(rama_gmm.means_, dtype=torch.float32)
RAMA_PREC = torch.tensor(np.linalg.inv(rama_gmm.covariances_), dtype=torch.float32)
RAMA_W = torch.tensor(rama_gmm.weights_ / (2 * np.pi * np.sqrt(np.linalg.det(rama_gmm.covariances_))), dtype=torch.float32)


def rama_score(phi, psi):
    """ln(R + C) per residue, R = mixture density of (φ, ψ) in radians, C = e^−10 (the paper's floor)."""
    d = torch.stack([phi, psi], -1)[:, None, :] - RAMA_MU[None]
    d = (d + math.pi) % (2 * math.pi) - math.pi                                   # angles wrap around
    q = torch.einsum("rki,kij,rkj->rk", d, RAMA_PREC, d)
    return torch.log((RAMA_W[None] * torch.exp(-0.5 * q)).sum(1) + math.exp(-10))


g = np.radians(np.linspace(-180, 180, 121))
GPHI, GPSI = np.meshgrid(g, g)
dens = np.exp(rama_gmm.score_samples(np.stack([GPHI.ravel(), GPSI.ravel()], 1))).reshape(GPHI.shape)
fig, ax = plt.subplots(figsize=(4.4, 4))
ax.contourf(np.degrees(GPHI), np.degrees(GPSI), np.log(dens + 1e-6), levels=15, cmap="Blues")
ax.scatter(np.degrees(ph.numpy()), np.degrees(ps.numpy()), color="tab:red", s=12, label="our model's residues")
ax.set_xlabel("φ (°)"); ax.set_ylabel("ψ (°)"); ax.legend(fontsize=7, loc="lower right")
ax.set_title("Ramachandran mixture (synthetic reference)", fontsize=9); plt.tight_layout(); plt.show()
print("Ramachandran score ln(R + C) of the model's residues: mean", f"{rama_score(ph, ps).mean().item():.2f}")

🔍 The helix residues sit in the α region; the turn residues sit in allowed regions too (β/PPII and left-handed helix).
A residue pushed into an empty region would get a score near $\ln C = -10$ — a strong penalty.

---
## 3 · Refining atoms into a map

Start from the Δ = 0° model and the Δ = +40° map. Let every atom move ($\mathbf x = \mathbf x_0 + \mathbf d$) and minimise

$$\mathcal L = -\,\mathrm{FRC}\big(\text{model projections},\ \text{map projections}\big) \;+\; w\cdot E_{\text{geometry}},$$

$$E_{\text{geometry}}=\overline{\Big(\frac{\ell-\ell_0}{0.02\,\text{Å}}\Big)^2}+\overline{\Big(\frac{\alpha-\alpha_0}{2°}\Big)^2}
  +10\sum_{\text{clashes}}(d_{\text{clash}}-d)^2-\overline{\ln(R+C)},$$

*in words:* match the map's projections, while keeping bonds ($\ell$) and angles ($\alpha$) near their ideal values,
atoms apart, and φ/ψ in allowed regions. Because we built the target ourselves, we can also measure what really matters:
the **RMSD to the true Δ = +40° model**. Three weights $w$: none, balanced, too strong.

In [ ]:
# ── 3.1 Refinement with three geometry weights ──
def geometry_energy(xyz):
    """The restraint energy E_geometry of a model (5n × 3)."""
    bl = (xyz[BONDS[:, 0]] - xyz[BONDS[:, 1]]).norm(dim=1)
    v1, v2 = xyz[ANGLES[:, 0]] - xyz[ANGLES[:, 1]], xyz[ANGLES[:, 2]] - xyz[ANGLES[:, 1]]
    ang = torch.acos(((v1 * v2).sum(1) / (v1.norm(dim=1) * v2.norm(dim=1))).clamp(-0.9999, 0.9999))
    D = (xyz[:, None] - xyz[None]).norm(dim=2) + 10 * torch.eye(len(xyz))
    clash = torch.relu(CLASH_D - D)[FAR]
    phi, psi = phi_psi(xyz)
    return (((bl - B0) / 0.02) ** 2).mean() + (((ang - A0) / math.radians(2.0)) ** 2).mean() \
        + 10 * (clash ** 2).sum() - rama_score(phi, psi).mean()


def rmsd(a, b) -> float:
    return float((a - b).pow(2).sum(1).mean().sqrt())


def refine_atoms(weight: float, steps: int = 300, target: float = 40.0):
    """Gradient refinement of all atom positions against one map; returns the refined coordinates."""
    torch.manual_seed(SEED)
    d = torch.zeros_like(x_start, requires_grad=True)
    opt = torch.optim.Adam([d], lr=0.05)
    for step in range(steps):
        x = x_start + d
        loss = -map_fit(x, target) + weight * geometry_energy(x)
        opt.zero_grad(); loss.backward(); opt.step()
    return (x_start + d).detach()


print(f"start (Δ = 0° model): RMSD to the true +40° model {rmsd(x_start, x_target):.2f} Å")
refined = {}
for w in [0.0, 1e-3, 1e-1]:
    t0 = time.time()
    refined[w] = refine_atoms(w)
    rep = geometry_report(refined[w])
    print(f"w = {w:<6g}: map FRC {map_fit(refined[w], 40.0).item():.3f}, RMSD to truth {rmsd(refined[w], x_target):.2f} Å, "
          f"bond RMSD {rep['bond_rmsd']:.3f} Å, angle RMSD {rep['angle_rmsd_deg']:.1f}°, clashes {rep['clashes']} "
          f"[{time.time() - t0:.0f} s]")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, (w, x) in zip(axes, refined.items()):
    ca = x.reshape(N_RES, 5, 3)[:, 1].numpy()
    ax.plot(*models[40.0][:, 1, :2].T, color="0.7", lw=4, label="truth (+40°)")
    ax.plot(ca[:, 0], ca[:, 1], "-o", ms=2, color="tab:red", label="refined")
    ax.set_aspect("equal"); ax.set_title(f"w = {w:g}", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without restraints ($w=0$) the atoms chase the density individually: the map fit is good but
the "model" is chemical nonsense — bonds of the wrong length, clashing atoms — and it is not even close to the true
structure, because at 4 Å a map cannot place individual atoms. With a balanced weight the map moves the helix as a whole
while chemistry keeps it a helix: the refined model lands within about an ångström of the truth with near-ideal
geometry. With too strong a weight the restraints win and the model barely moves. Chen (2025) sets the weight
automatically from the ratio of the map gain to the geometry loss.

✏️ **Your turn.** Try `w = 1e-2` and `w = 1e-4`. Where is the sweet spot? Does it move if you refine against the
Δ = +20° map instead?

---
## 4 · Patches, residues, atoms

For a large complex, moving every atom independently from the start is slow and fragile: a domain that must travel
10 Å is pulled atom by atom. Chen (2025) refines in **stages whose outputs add up** ("boosting-like"): first rigid
transforms of **patches** (64 patches from K-means on residue centres), then **per-residue** corrections, then **per-atom**
corrections — each produced by a small dense network with a constant input. A network with a constant input is simply
a flexible way to hold parameters; in our toy we hold them directly. We use 4 patches and compare with atoms-only
refinement at the same number of steps.

In [ ]:
# ── 4.1 Staged refinement: 4 rigid patches → per-residue shifts → per-atom shifts ──
def axis_angle_matrix_torch(v):
    """Rotation matrices (b, 3, 3) from axis-angle vectors (b, 3)."""
    angle = v.norm(dim=1, keepdim=True).clamp_min(1e-8)
    k = v / angle
    K = torch.zeros(v.shape[0], 3, 3)
    K[:, 0, 1], K[:, 0, 2], K[:, 1, 2] = -k[:, 2], k[:, 1], -k[:, 0]
    K[:, 1, 0], K[:, 2, 0], K[:, 2, 1] = k[:, 2], -k[:, 1], k[:, 0]
    s, c = torch.sin(angle)[..., None], torch.cos(angle)[..., None]
    return torch.eye(3)[None] + s * K + (1 - c) * K @ K


ca0 = models[0.0][:, 1]
patch_centres = ca0[np.linspace(0, N_RES - 1, 4).astype(int)].copy()
for _ in range(20):                                                      # K-means on residue (Cα) centres
    lab_res = np.argmin(((ca0[:, None] - patch_centres[None]) ** 2).sum(-1), 1)
    patch_centres = np.array([ca0[lab_res == k].mean(0) for k in range(4)])
PATCH_OF_ATOM = torch.tensor(np.repeat(lab_res, 5))
CENT_T = torch.tensor(patch_centres, dtype=torch.float32)
RES_OF_ATOM = torch.tensor(res_of)


def staged_refine(weight: float = 1e-3, steps: int = 300):
    """Three stages (100 steps each); every stage adds its correction to the previous ones."""
    torch.manual_seed(SEED)
    rot, trans = torch.zeros(4, 3, requires_grad=True), torch.zeros(4, 3, requires_grad=True)
    res_shift = torch.zeros(N_RES, 3, requires_grad=True)
    atom_shift = torch.zeros_like(x_start, requires_grad=True)

    def coords():
        R = axis_angle_matrix_torch(rot)[PATCH_OF_ATOM]
        c = CENT_T[PATCH_OF_ATOM]
        x = torch.einsum("aij,aj->ai", R, x_start - c) + c + trans[PATCH_OF_ATOM]      # stage 1: rigid patches
        return x + res_shift[RES_OF_ATOM] + atom_shift                                    # stages 2 and 3

    history = []
    for stage, params in enumerate([[rot, trans], [res_shift], [atom_shift]]):
        opt = torch.optim.Adam(params, lr=0.02 if stage == 0 else 0.05)
        for step in range(steps // 3):
            x = coords()
            loss = -map_fit(x, 40.0) + weight * geometry_energy(x)
            opt.zero_grad(); loss.backward(); opt.step()
        history.append(rmsd(coords().detach(), x_target))
    return coords().detach(), history


t0 = time.time()
x_staged, hist = staged_refine()
rep = geometry_report(x_staged)
print(f"RMSD to truth after each stage: patches {hist[0]:.2f} Å → residues {hist[1]:.2f} Å → atoms {hist[2]:.2f} Å "
      f"(atoms-only refinement: {rmsd(refined[1e-3], x_target):.2f} Å)  [{time.time() - t0:.0f} s]")
print("geometry after the staged refinement:", rep)

🔍 **What just happened?** The rigid patches do the heavy lifting — the second helix is (mostly) one patch and simply
turns into place — and the residue and atom stages polish. With the same number of steps, the staged refinement ends
much closer to the truth than moving atoms one by one from the start, because no atom-by-atom tug-of-war is needed to
move a whole helix. With tens of thousands of atoms, large motions and noisy maps, the coarse stages keep domains intact
and make convergence fast (the author reports single refinements in under an hour).

---
## 5 · A model series

Now the main idea of the paper. A heterogeneity method gives a *series* of maps along a latent coordinate $z$ (from
cryoDRGN, 3DFlex, 3DVA or a GMM network — Modules 26–28). Chen (2025) trains a network **from $z$ to atom coordinates**
so that the model at each $z_k$ fits map $k$, and adds the geometry restraints at **uniformly spaced latent values**,
including values *between* the maps — so that every model along the path is chemically sound. (The paper also perturbs
the input randomly and trains in stages.)

Our series: the five maps with latent values $z = \Delta/40° \in \{-1, -0.5, 0, 0.5, 1\}$. The network outputs a
displacement of all 180 atoms from the Δ = 0° model. We then ask for models at $z = \pm0.25, \pm0.75$ — conformations
for which **no map** was given — and compare them with the true models at Δ = ±10°, ±30°.

In [ ]:
# ── 5.1 Train a latent → atoms network on the five maps ──
Z_OF = {d: d / 40.0 for d in DELTAS}
torch.manual_seed(SEED)
series_net = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(),
                           nn.Linear(64, x_start.numel()))
nn.init.zeros_(series_net[-1].weight); nn.init.zeros_(series_net[-1].bias)          # start: every z → the Δ = 0° model
opt = torch.optim.Adam(series_net.parameters(), lr=1e-3)
model_at = lambda z: x_start + series_net(torch.tensor([[z]], dtype=torch.float32)).reshape(-1, 3)
t0 = time.time()
for step in range(1500):
    d = DELTAS[step % len(DELTAS)]
    x = model_at(Z_OF[d])
    z_between = float(np.random.default_rng(step).uniform(-1, 1))                       # restraints between the maps too
    loss = -map_fit(x, d) + 1e-3 * geometry_energy(x) + 1e-3 * geometry_energy(model_at(z_between))
    opt.zero_grad(); loss.backward(); opt.step()
print(f"model-series network trained in {time.time() - t0:.0f} s")

In [ ]:
# ── 5.2 Models along the series — at the maps and between them — vs naïve interpolation ──
print(f"{'Δ (°)':>6s} {'z':>6s} {'map given?':>11s} {'RMSD to truth':>14s} {'bond RMSD':>10s} {'clashes':>8s}")
series_rows = []
with torch.no_grad():
    for d in [-40.0, -30.0, -20.0, -10.0, 0.0, 10.0, 20.0, 30.0, 40.0]:
        x = model_at(d / 40.0)
        truth_d = torch.tensor(model(d).reshape(-1, 3), dtype=torch.float32)
        rep = geometry_report(x)
        series_rows.append((d, rmsd(x, truth_d), rep["bond_rmsd"]))
        print(f"{d:6.0f} {d / 40:6.2f} {'yes' if d in DELTAS else 'no':>11s} {rmsd(x, truth_d):12.2f} Å "
              f"{rep['bond_rmsd']:8.3f} Å {rep['clashes']:8d}")
x_lin = 0.5 * (torch.tensor(models[-40.0].reshape(-1, 3)) + torch.tensor(models[40.0].reshape(-1, 3))).float()
rep_lin = geometry_report(x_lin)
print(f"naïve average of the −40° and +40° models: RMSD to the true 0° model {rmsd(x_lin, x_start):.2f} Å, "
      f"bond RMSD {rep_lin['bond_rmsd']:.3f} Å, clashes {rep_lin['clashes']}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
with torch.no_grad():
    for d, col in zip(np.linspace(-40, 40, 9), plt.cm.coolwarm(np.linspace(0, 1, 9))):
        ca = model_at(d / 40).reshape(N_RES, 5, 3)[:, 1].numpy()
        axes[0].plot(ca[:, 0], ca[:, 1], color=col, lw=1.5)
axes[0].set_aspect("equal"); axes[0].set_title("Cα traces of the model series (z from −1 to +1)", fontsize=9)
axes[0].set_xticks([]); axes[0].set_yticks([])
ds, r_, b_ = np.array(series_rows).T
given = np.isin(ds, DELTAS)
axes[1].plot(ds[given], r_[given], "o", color="tab:blue", label="RMSD, map given")
axes[1].plot(ds[~given], r_[~given], "s", color="tab:orange", label="RMSD, no map (between)")
axes[1].axhline(rmsd(x_lin, x_start), color="tab:red", ls="--", lw=1, label="naïve interpolation at 0°")
axes[1].set_xlabel("hinge angle Δ (°)"); axes[1].set_ylabel("RMSD to the true model (Å)"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** One small network now *is* a continuous family of atomic models. At the five maps it
reproduces the true conformations closely, and — thanks to the restraints applied between the maps — the in-between
models are also close to the true intermediate structures and chemically sound. Simply averaging the coordinates of two
end states, by contrast, cuts across the arc of a rotation: bonds stretch and squash, atoms clash, and the result is not
the structure in the middle. A model series is a *hypothesis about the path*, constrained by chemistry and by every map
along the way.

---
## 6 · What a model series tells you — and what it does not

- ✅ **Distances and contacts along a motion**: which residues come together, how far a helix travels — measurable on
  models, not on blobs.
- ✅ **Consistency checks**: geometry should stay good everywhere; a model that needs bad geometry to fit a map points to
  a problem in the map or the latent coordinate.
- ⚠️ **Static chemistry only.** The restraints say what a single structure may look like, not how fast or in which order
  things move; there is no force field (no energies, no kinetics) — a limitation the author states.
- ⚠️ **Resolution along the series** is chosen by eye (the paper fits to 7 Å for the TRPV1 series and 15 Å for the
  spliceosome series); ligands and nucleic acids are guided mostly by the maps.
- ⚠️ **Validate** each frame against its map (map–model FSC, Q-scores) and remember that the maps themselves come from a
  heterogeneity method with its own assumptions (Modules 26–28).

### 📄 Paper connection — Chen 2025: molecular model series from heterogeneous cryo-EM structures

**Paper:** Chen M. *Building molecular model series from heterogeneous CryoEM structures using Gaussian mixture models
and deep neural networks.* Communications Biology 8:798 (2025). [doi:10.1038/s42003-025-08202-9](https://doi.org/10.1038/s42003-025-08202-9)
(preprint [bioRxiv 10.1101/2024.09.27.615511](https://doi.org/10.1101/2024.09.27.615511))

- **Question it answers:** how to turn single maps — and whole series of maps from heterogeneity analyses — into
  chemically valid atomic models.
- **What the paper actually does:** one Gaussian per non-hydrogen atom (five parameters each); the data term is the FRC
  between projections of the GMM and of the map; geometry terms: bond/angle Gaussian likelihoods with an outlier penalty at
  4.5σ, planarity thresholds (30° peptide, 10° others), clashes below the van der Waals sum minus 0.4 Å (+0.4 Å for
  H-bonds), Ramachandran 2D GMMs (six residue classes, 2,500 Gaussians each) scored as ln(R + C) with C = e⁻¹⁰, rotamer
  GMMs (4/25/64/160 Gaussians for 1–4 χ angles) and an RNA suite library; the geometry weight comes from the ratio of
  map gain to geometry loss. Single models: four-layer dense decoders with a constant input, three networks in sequence
  whose outputs are summed — 64 patch transforms (K-means on residue centres), residue level, full atom. Series: input =
  the stack of maps with their latent values from any heterogeneity method (cryoDRGN, 3DFlex, 3DVA, GMM-DNN); random
  input perturbation; staged training; extra geometry refinement at uniformly spaced latent values; FSC to a chosen
  cutoff (3 Å for single models, 7 Å for the TRPV1 series, 15 Å for the spliceosome series).
- **Result reported:** geometry improved for all 25 benchmark entries with Q-scores about unchanged; single refinements
  in under an hour; a series for the >100,000-atom spliceosome (EMPIAR-10180, from PDB 5NRL) in about 5 hours; a TRPV1
  series from EMPIAR-10059 (model deposited as PDB 9OGK).
- **What our toy keeps:** one Gaussian per atom, FRC of projections, bond/angle/clash/Ramachandran restraints (simplified),
  the weight trade-off, staged patch → residue → atom refinement, a latent → atoms series network with restraints between
  the maps.
- **What our toy leaves out:** side chains, rotamers, planarity, RNA, automatic weighting, real maps.

### 📄 Paper connection — Chen, Toader & Lederman 2023: models inside deep GMMs

The 2023 paper (Modules 25–26) already started GMMs **from atomic models** — pre-training the decoder so its output
coordinates match the model's atoms — and added a small restraint on Cα–Cα distances and pseudo-bond angles to keep
models sensible during heterogeneity training. Chen (2025) completes that path with full stereochemistry and series.

---
## ✅ Checkpoint

1. Which two angles set the shape of a protein backbone, and which one is held flat?
   <details><summary>Answer</summary>φ (about N–Cα) and ψ (about Cα–C); ω, the peptide bond torsion, stays near 180°.</details>
2. Why does refinement without restraints give a good map fit but a bad model?
   <details><summary>Answer</summary>At 4 Å the map cannot place single atoms; free atoms rearrange to fit density in ways that break bonds and create clashes. Chemistry supplies the missing information.</details>
3. What does the Ramachandran score ln(R + C) do for residues in forbidden regions?
   <details><summary>Answer</summary>R is near 0, so the score is near ln C = −10: a large penalty that pushes φ/ψ back toward allowed regions, while C keeps the logarithm finite.</details>
4. What is the benefit of patch → residue → atom stages?
   <details><summary>Answer</summary>Large rigid motions are captured with few parameters first, keeping domains intact; finer stages only correct details.</details>
5. Why apply geometry restraints at latent values between the maps?
   <details><summary>Answer</summary>No map constrains those models; without restraints the network could produce chemically impossible intermediates. Restraints make every model along the path valid.</details>
6. Why is averaging two end-state models a bad intermediate?
   <details><summary>Answer</summary>Rotations move atoms on arcs; averaging coordinates follows the chord, shortening and distorting bonds and producing clashes.</details>

## 🧠 Questions & discussion

1. A model series shows a salt bridge forming halfway through a motion. What extra evidence would you want before
   writing that the salt bridge *drives* the motion?
2. The latent coordinate came from a heterogeneity method. If two different methods give different latent orders of the
   same maps, what happens to the model series?
3. Why might Q-scores stay about the same while geometry improves (as the paper reports)?
4. Compare a model series with molecular dynamics: what does each know that the other does not?

## 🚀 Bonus — going further

- **Noisy maps.** Add Gaussian noise to the five maps (Module 04) before projecting, and repeat Sections 3 and 5.
- **A wrong weight.** Train the series with geometry only at the five map values (drop the `z_between` term). What
  happens to the in-between models?
- **Side chains.** Extend the model with the toolkit's schematic side chains (Module 01) and add their bonds.
- **Real software:** read the paper's methods with Sections 2–5 at hand; EMAN2 hosts the GMM programs
  (<https://github.com/cryoem/eman2>).

## 📝 Lab note

```text
Module 30 — date:
Input: code-built 36-residue helix–turn–helix (N, CA, C, O, CB), five 4 Å maps along a hinge (Δ = −40° … +40°)
Output: refined models (three restraint weights), staged refinement, a latent → atoms model series
Method: one Gaussian per atom, FRC of projections, bond/angle/clash/Ramachandran restraints, patch → residue → atom stages
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Chen 2025 (Commun Biol); Chen, Toader & Lederman 2023
```

## 📚 References

1. Chen M. Building molecular model series from heterogeneous CryoEM structures using Gaussian mixture models and deep neural networks. *Commun Biol* 8:798 (2025). doi:10.1038/s42003-025-08202-9
2. Chen M, Toader B, Lederman R. Integrating molecular models into CryoEM heterogeneity analysis using scalable high-resolution deep Gaussian mixture models. *J Mol Biol* 435(9):168014 (2023). doi:10.1016/j.jmb.2023.168014
3. Engh RA, Huber R. Accurate bond and angle parameters for X-ray protein structure refinement. *Acta Cryst A* 47:392–400 (1991). doi:10.1107/S0108767391001071
4. Ramachandran GN, Ramakrishnan C, Sasisekharan V. Stereochemistry of polypeptide chain configurations. *J Mol Biol* 7:95–99 (1963). doi:10.1016/S0022-2836(63)80023-6
5. Parsons J *et al.* Practical conversion from torsion space to Cartesian space for in silico protein synthesis. *J Comput Chem* 26:1063–1068 (2005). doi:10.1002/jcc.20237 — the NeRF chain builder.
6. Pintilie G *et al.* Measurement of atom resolvability in cryo-EM maps with Q-scores. *Nat Methods* 17:328–334 (2020). doi:10.1038/s41592-020-0731-1

## ➡️ Next

**Part G begins: [31 · Picking particles with almost no labels](31_picking_with_sparse_labels.ipynb)** — back to the cell,
and the frontiers of finding molecules with very few annotations.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")